### Statistical Decision Support
Statistical Decision Support is designed to solve a core retail operational problem— `balancing product supply with customer demand` using data science instead of guesswork.

Many retail companies struggle with stockouts (running out of popular items) or holding excess inventory (tying up cash in unsold stock). This project builds a statistical engine using Python to analyze raw store records and drive optimal inventory decisions. This is achieved by using the techniques of the ___mathematics and statistics___ in supporting the business plans and efforts... The results becomes indeed impactful and profitable for businesses when they do apply the math+stat combination ___rather than random guessing or synthesizing effortless hypothesis___!


### A referral Example -
It is believed in a running business(say, ###Retail Company) that Promotions increases Sales! Well, this might be highly relevant, but the statement would only be backed when it had caused observable results...

So, the Data-Scientists must then **explore the statistical evidences** to rather back up the statement, and if somehow this becomes valid, then we ought to say that:

`"The statement is statically supported"`. That is what we are going to do here..

### Business Hypothesis:
Retail Promotions increase the number of units sold.

**Null Hypothesis** - $H_0$ 
 Promotions do not increase average units sold.

**Alternative hypothesis** — $H_1$ 
 Promotional periods have a higher average number of units sold than non-promotional periods.


In [1]:
import pandas as pd
import numpy as np
from scipy import stats

test_dataset = pd.read_csv("../Datasets_MLmodels/Demand/P02_outputs/processed_inventory_data.csv")
test_dataset.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,...,Competitor Pricing,Seasonality,Year,Month,Day,Inventory Status,Inventory Utilization,Order Difference,Sales Difference,Demand Status
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,...,29.69,Autumn,2022,1,1,Medium,54.98,-80.47,-8.47,Lower than Forecast
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,...,66.16,Autumn,2022,1,1,Medium,73.53,-78.04,5.96,Higher than Forecast
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,...,31.32,Summer,2022,1,1,Medium,63.73,-23.02,-9.02,Lower than Forecast
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,...,34.74,Autumn,2022,1,1,Healthy,13.01,101.82,-1.18,Lower than Forecast
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,...,68.95,Summer,2022,1,1,Medium,8.43,125.74,4.74,Higher than Forecast


In [2]:
test_dataset.shape

(73100, 23)

In [3]:
#test_dataset['Holiday/Promotion']
for i in range(0,23):
    if test_dataset.columns[i] == 'Holiday/Promotion':
        print(i)

12


In [4]:
test_dataset.columns[12]

'Holiday/Promotion'

In [5]:
test_dataset['Holiday/Promotion'].value_counts()

Holiday/Promotion
0    36747
1    36353
Name: count, dtype: int64

In [6]:
required_columns = ["Holiday/Promotion", "Units Sold"]

for column in required_columns:
    if column not in test_dataset.columns:
        raise ValueError(f"Required column not found in dataset: {column}")
# This has to be performed because, the hypothesis compares the sales during the promotional/ non-promotional periods...
#  By conforming the presence of the valid columns, we could easily proceed!!

In [7]:
analysis_data = test_dataset[["Holiday/Promotion", "Units Sold"]].copy()
analysis_data = analysis_data.dropna()

print("\nRecords available for analysis:", len(analysis_data))


Records available for analysis: 73100


In [8]:
promotion_values = analysis_data["Holiday/Promotion"].unique()

promotion_sales = analysis_data[analysis_data["Holiday/Promotion"] == 1]["Units Sold"]
non_promotion_sales = analysis_data[analysis_data["Holiday/Promotion"] == 0]["Units Sold"]

In [9]:
units_sold_array = test_dataset['Units Sold'].unique()
print(units_sold_array.shape)
# Contains 498 unique values in Units Sold column!

(498,)


In [10]:
promotion_values

array([0, 1])

In [11]:
print("Promotional Records: ", len(promotion_sales))
print("Non-Promotional Records: ", len(non_promotion_sales))

Promotional Records:  36353
Non-Promotional Records:  36747


In [12]:
print("Average units sold during promotion:", promotion_sales.mean())
print("Average units sold without promotion:", non_promotion_sales.mean())

Average units sold during promotion: 136.4239264985008
Average units sold without promotion: 136.50537458840176


In [13]:
promotion_variance = promotion_sales.var()
non_promotion_variance = non_promotion_sales.var()

print("\nVARIANCE CHECK")
print("-" * 65)
print("Promotion variance:", promotion_variance)
print("Non-promotion variance:", non_promotion_variance)


VARIANCE CHECK
-----------------------------------------------------------------
Promotion variance: 11785.01011962848
Non-promotion variance: 11941.34261384956


In [14]:
levene_stat, levene_p = stats.levene(
    promotion_sales,
    non_promotion_sales
)

print("\nLEVENE'S TEST")
print("-" * 65)

print("Test statistic:", levene_stat)
print("p-value:", levene_p)

if levene_p < 0.05:
    print("Result: The variances are significantly different.")
    print("Therefore, Welch's t-test is appropriate.")
else:
    print("Result: No statistically significant variance difference detected.")
    print("Welch's t-test will still be used because it is more robust.")    


LEVENE'S TEST
-----------------------------------------------------------------
Test statistic: 0.6377683734513794
p-value: 0.42452265316018156
Result: No statistically significant variance difference detected.
Welch's t-test will still be used because it is more robust.


In [15]:
print("\nNORMALITY CHECK")
print("-" * 65)

sample_size = min(5000, len(promotion_sales))

promotion_sample = promotion_sales.sample(
    sample_size,
    random_state=42
)

non_promotion_sample = non_promotion_sales.sample(
    min(5000, len(non_promotion_sales)),
    random_state=42
)

promotion_shapiro_stat, promotion_shapiro_p = stats.shapiro(promotion_sample)
non_promotion_shapiro_stat, non_promotion_shapiro_p = stats.shapiro(non_promotion_sample)

print("Promotion sample Shapiro p-value:", promotion_shapiro_p)
print("Non-promotion sample Shapiro p-value:", non_promotion_shapiro_p)
print(
    "\nBecause the dataset contains many observations, "
    "the t-test is reasonably robust to moderate departures "
    "from normality."
    )


NORMALITY CHECK
-----------------------------------------------------------------
Promotion sample Shapiro p-value: 4.3466920638106624e-46
Non-promotion sample Shapiro p-value: 5.810046276941326e-48

Because the dataset contains many observations, the t-test is reasonably robust to moderate departures from normality.


In [16]:
test_statistic, p_value = stats.ttest_ind(
    promotion_sales,
    non_promotion_sales,
    equal_var=False,
    alternative="greater"
)

print("\nWELCH'S T-TEST")
print("-" * 65)

print("Test statistic:", test_statistic)
print("p-value:", p_value)


WELCH'S T-TEST
-----------------------------------------------------------------
Test statistic: -0.10109053770851395
p-value: 0.5402605666789482


In [17]:
alpha = 0.05

print("\nSignificance level:", alpha)


Significance level: 0.05


In [18]:
if p_value < alpha:
    statistical_result = "Statistically significant"

    print("\nResult: Reject the null hypothesis.")
    print("There is statistical evidence that promotions increase average units sold.")
else:
    statistical_result = "Not statistically significant"

    print("\nResult: Fail to reject the null hypothesis.")
    print("There is not enough statistical evidence to conclude that promotions increase average units sold.")    


Result: Fail to reject the null hypothesis.
There is not enough statistical evidence to conclude that promotions increase average units sold.


In [19]:
promotion_mean = promotion_sales.mean()
non_promotion_mean = non_promotion_sales.mean()

mean_difference = (promotion_mean - non_promotion_mean)

In [20]:
if non_promotion_mean != 0:
    percentage_difference = (mean_difference/non_promotion_mean) * 100
else:
    percentage_difference = 0

print("\nBUSINESS EFFECT")
print("-" * 65)

print("Difference in average units sold:", mean_difference)
print("Percentage difference:", percentage_difference, "%")


BUSINESS EFFECT
-----------------------------------------------------------------
Difference in average units sold: -0.08144808990095953
Percentage difference: -0.059666581002063936 %


In [21]:
promotion_std = promotion_sales.std()
non_promotion_std = non_promotion_sales.std()

n1 = len(promotion_sales)
n2 = len(non_promotion_sales)

pooled_std = np.sqrt(
    ( ((n1 - 1) * promotion_std ** 2) + ((n2 - 1) * non_promotion_std ** 2) ) / (n1 + n2 - 2) 
)

if pooled_std != 0:
    cohens_d = (promotion_mean - non_promotion_mean) / pooled_std

else:
    cohens_d = 0
print("Cohen's d:", cohens_d)

Cohen's d: -0.0007477780252495794


In [22]:
if abs(cohens_d) < 0.2:
    effect_description = "Small"

elif abs(cohens_d) < 0.5:
    effect_description = "Small to moderate"

elif abs(cohens_d) < 0.8:
    effect_description = "Moderate"
else:
    effect_description = "Large"

print("Effect size:", effect_description)

Effect size: Small


In [23]:
results = pd.DataFrame({
    "Metric": [
        "Promotion sample size","Non-promotion sample size","Promotion average units sold","Non-promotion average units sold",
        "Mean difference", "Percentage difference", "Levene p-value", "Welch t-statistic", "Welch t-test p-value",
        "Significance level", "Cohen's d", "Effect size", "Statistical result"
    ],
    
    "Value": [
        len(promotion_sales), len(non_promotion_sales), promotion_mean, non_promotion_mean, mean_difference,  
        percentage_difference, levene_p, test_statistic, p_value, alpha, cohens_d, effect_description, statistical_result
    ]
})
# Below is an example of how a tabular data is mapped/ created using dictionary and lists!!
owner_to_spells_data = ({'Owner': ['Harry', 'Voldemort', 'Bella'],
                   'Spells': ['Expecto-Patronum', 'Avada-Kedavra', 'Crucio']})
Created_dataset = pd.DataFrame(owner_to_spells_data)
Created_dataset

,Owner,Spells
0,Harry,Expecto-Patronum
1,Voldemort,Avada-Kedavra
2,Bella,Crucio


In [24]:
results.to_csv("../Datasets_MLmodels/Demand/P05_outputs/statistical_results.csv", index=False)

In [25]:
if p_value < alpha:
    conclusion = (
        "The statistical test provides evidence that promotional periods have higher average units sold than non-promotional"
        "periods. The retailer can therefore consider promotions when planning inventory, while also considering the magnitude" 
        "of the effect and other business factors."
    )
else:
    conclusion = (
        "The statistical test does not provide sufficient evidence that promotional periods have higher average units sold"
        "than non-promotional periods. Promotions should therefore not be assumed to require additional inventory solely" 
        "on the basis of this analysis."
    )    

In [26]:
with open("../Datasets_MLmodels/Demand/P05_outputs/business_conclusion.txt", "w") as f:
    f.write("PROJECT 05 - BUSINESS CONCLUSION\n")
    f.write("=" * 50 + "\n\n")

    f.write("Business Question:\n")
    f.write("Does running a promotion have a statistically significant positive effect on units sold?\n\n")

    f.write("Null Hypothesis:\n")
    f.write("Promotions do not increase average units sold.\n\n")

    f.write("Alternative Hypothesis:\n")
    f.write("Promotions increase average units sold.\n\n")

    f.write("Statistical Test:\n")
    f.write("Welch's independent two-sample t-test\n\n")

    f.write("p-value:\n")
    f.write(str(p_value) + "\n\n")

    f.write("Significance Level:\n")
    f.write(str(alpha) + "\n\n")

    f.write("Average Units Sold During Promotion:\n")
    f.write(str(promotion_mean) + "\n\n")

    f.write("Average Units Sold Without Promotion:\n")
    f.write(str(non_promotion_mean) + "\n\n")

    f.write("Percentage Difference:\n")
    f.write(str(percentage_difference) + "%\n\n")

    f.write("Cohen's d:\n")
    f.write(str(cohens_d) + "\n\n")

    f.write("Effect Size:\n")
    f.write(effect_description + "\n\n")

    f.write("Conclusion:\n")
    f.write(conclusion)